# AdMarket Arena — Advertiser GRPO Training

Trains the **trained LLM advertiser** (Theme 1 + Theme 2 headline) on the AdMarket Arena env via TRL GRPO + Unsloth 4-bit LoRA.

**T4 (Colab) tuned config** — base model: Qwen2.5-3B-Instruct (4-bit NF4 + LoRA r=16), GRPO `num_generations=4`, 64-token output cap, ~4 hr expected for 50-80 GRPO steps. Curriculum scheduler escalates `arena_easy → arena_medium → arena_hard` when mean episode reward > 0.30 for 10 consecutive rollouts (master Section 3.1.1).

Reuses Plan 3's shared `training_callbacks.py` (W&B + CSV + JSONL + best-checkpoint tracker) with `wandb_project='admarket-arena-advertiser'`. Plan 2's `train_oversight.ipynb` reuses the same callback with a different project name.

**Outputs**:
- `checkpoints/advertiser_run/checkpoint-<N>/` LoRA adapters (every 10 GRPO steps)
- `checkpoints/advertiser_run/best/` copy of the best by mean weekly_roas over last 5 validation episodes
- `logs/training_run_advertiser_grpo_*.csv` tamper-evident receipt
- `logs/episodes/advertiser_grpo/ep_step*.jsonl` validation episode dumps every 10 steps
- W&B project `admarket-arena-advertiser` (public)
- HF Hub push to `<org>/admarket-advertiser-qwen2.5-3b-grpo`

**Reward = episode return**: per-step engagement + daily ROAS bonus + weekly ROAS bonus (master Section 4). Per master Section 6, one prompt = one rollout step (the agent emits a single `AuctionAction` JSON per impression opportunity); the env supplies the multi-step composition under the hood.

## 1. Install dependencies

In [ ]:
%%capture
# Pin versions tested with Unsloth on Colab T4 (Apr 2026) — same stack as train_oversight.ipynb so we share the cached env.
!pip install -U unsloth
!pip install -U trl==0.12.* peft==0.13.* transformers==4.46.* accelerate==1.1.* datasets==3.* bitsandbytes==0.44.*
!pip install -U wandb pydantic matplotlib

## 2. Mount drive / clone repo

If you've copied the repo into the Colab runtime already, skip the clone and just `cd` into it. Otherwise pull from the team repo so the latest `competitors.py`, `training_callbacks.py`, `curriculum_scheduler.py`, `scripts/advertiser_eval.py` are available.

In [ ]:
import os, sys
REPO_PATH = '/content/meta_ad_optimizer'  # change to your local path if running outside Colab
if not os.path.exists(REPO_PATH):
    !git clone https://github.com/<your-org>/meta_ad_optimizer.git $REPO_PATH
%cd $REPO_PATH
if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)

## 3. Imports + config

In [ ]:
import json
import random
import statistics
import textwrap
from pathlib import Path
from typing import Dict, List

import torch
import wandb
from datasets import Dataset

from competitors import (
    ADVERTISER_SYSTEM_PROMPT,
    PERSONAS,
    PersonaBot,
    _format_observation_for_advertiser,
    parse_llm_advertiser_action,
)
from curriculum_scheduler import make_advertiser_curriculum
from models import AuctionAction, AuctionObservation
from tasks import ARENA_TASKS
from training_callbacks import make_arena_callback

# --- knobs ---
BASE_MODEL          = 'unsloth/Qwen2.5-3B-Instruct-bnb-4bit'
MAX_SEQ_LEN         = 4096
LORA_RANK           = 16
OUTPUT_DIR          = Path('checkpoints/advertiser_run')
MAX_NEW_TOKENS      = 64           # action JSON only (master Section 6 T4 budget)
NUM_GENERATIONS     = 4            # GRPO group size, T4-tight
LEARNING_RATE       = 1e-5
MAX_STEPS           = 80           # ~4 hr at ~3 min/step on T4 with arena_hard rollouts
SAVE_EVERY_STEPS    = 10
LOG_EVERY_STEPS     = 1
WANDB_PROJECT       = 'admarket-arena-advertiser'
RUN_NAME            = 'advertiser_grpo_qwen3b'
HF_HUB_REPO_ID      = '<your-org>/admarket-advertiser-qwen2.5-3b-grpo'  # set before push
SEED                = 42

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
Path('logs').mkdir(parents=True, exist_ok=True)
Path('logs/episodes').mkdir(parents=True, exist_ok=True)

random.seed(SEED); torch.manual_seed(SEED)
print('Config OK.  current curriculum tier =', 'arena_easy')

## 4. Load model with Unsloth (4-bit LoRA)

Same recipe as `train_oversight.ipynb` — shared base weights are cached on disk so the second notebook (oversight) loads in seconds when run sequentially after this one.

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_alpha=LORA_RANK,
    lora_dropout=0.0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=SEED,
)
tokenizer.pad_token = tokenizer.pad_token or tokenizer.eos_token
print(f'Model loaded: {sum(p.numel() for p in model.parameters() if p.requires_grad):,} trainable params (LoRA).')

## 5. Curriculum scheduler

The scheduler drives both the env tier (`arena_easy → arena_medium → arena_hard`) and the rollout config (impression count, opponent slate). Promotion fires when mean rollout reward > 0.30 for 10 consecutive rollouts. The TrainerCallback bridge (`scheduler.as_callback()`) feeds rollout rewards in automatically on every TRL `on_log` event so the notebook only has to call `add_callback` once.

In [ ]:
_current_tier = {'name': 'arena_easy'}

def _on_promote(old: str, new: str, step: int) -> None:
    print(f'[curriculum] step={step} promoting {old} -> {new}')
    _current_tier['name'] = new

scheduler = make_advertiser_curriculum(on_promote=_on_promote, promotion_threshold=0.30, required_streak=10)
print('Curriculum scheduler ready. Initial tier:', scheduler.current_tier)

## 6. Build dataset (one prompt = one auction step)

GRPO trains on (prompt, group of completions, reward) triples. Each prompt is a rendered `AuctionObservation` for the trained advertiser at one auction step; the reward function below is the **per-step engagement + daily/weekly ROAS** composite from master Section 4.

Synthetic-prompt fallback: until Plan 1's `arena_env.py` lands, we generate prompts by spinning up the same `scripts/advertiser_eval.run_episode` synthetic loop and emitting one (observation, ground-truth-reward) pair per step with the trained advertiser as a no-op (it never bids), then replacing its policy with the model under training. Once Plan 1 ships, swap the dataset builder for a real env rollout and the rest of the notebook is unchanged.

In [ ]:
from scripts.advertiser_eval import _MockTrainedPolicy, _sample_persona_slate, run_episode

BUILD_EXAMPLES_PER_TIER = 200

def _episode_to_examples(episode_seed: int, tier: str) -> List[Dict[str, str]]:
    """Run one synthetic episode and emit (prompt, recorded_observation_json) pairs
    for each step the trained advertiser participated in."""
    cfg = ARENA_TASKS[tier]
    rng = random.Random(episode_seed)
    persona_names = list(PERSONAS.keys())[:cfg.n_personas]
    opponents = _sample_persona_slate(rng, persona_names, n=cfg.n_personas, jitter_scale=1.0)
    captured: List[Dict[str, str]] = []

    def capturing_policy(observation: AuctionObservation, state=None) -> AuctionAction:
        prompt_user = _format_observation_for_advertiser(observation)
        chat = tokenizer.apply_chat_template(
            [
                {'role': 'system', 'content': ADVERTISER_SYSTEM_PROMPT},
                {'role': 'user', 'content': prompt_user},
            ],
            tokenize=False,
            add_generation_prompt=True,
        )
        captured.append({
            'prompt': chat,
            'observation_json': observation.model_dump_json(),
            'tier': tier,
        })
        # Use the deterministic mock policy so the synthetic env runs to completion;
        # the captured prompts are what TRL trains on.
        return _MockTrainedPolicy().bid(observation, state)

    run_episode(seed=episode_seed, cfg=cfg, trained_policy=capturing_policy, opponents=opponents)
    return captured

examples: List[Dict[str, str]] = []
for ep in range(BUILD_EXAMPLES_PER_TIER):
    examples.extend(_episode_to_examples(SEED + ep, scheduler.current_tier))
    if len(examples) >= 1000:
        break

random.Random(SEED).shuffle(examples)
split_at = max(1, int(len(examples) * 0.9))
train_ds = Dataset.from_list(examples[:split_at])
val_ds   = Dataset.from_list(examples[split_at:])
print(f'Built {len(examples)} step-examples ({scheduler.current_tier}).  train={len(train_ds)}  val={len(val_ds)}')
print(train_ds[0]['prompt'][:400], '...')

## 7. Reward function — per-step + daily/weekly ROAS proxy

Master Section 4 defines the full reward as `per_step_reward + daily_reward (boundaries) + weekly_reward (episode end)`. For per-step GRPO training we collapse this into a single scalar per (prompt, completion) by:

  - parsing the LLM's JSON action,
  - simulating the auction outcome against the recorded observation's competitive context (recent clearing prices + persona snapshot stored in the observation),
  - computing per-step engagement (clicks - clearing-price waste - skip nudge),
  - amortising the daily + weekly ROAS bonuses across the episode's steps.

When Plan 1 ships, replace this with `arena_env` calling `Rubric.score(...)` directly via `server/arena_rubrics.py` (the `OversightF1Rubric` template is already in that module).

In [ ]:
def _per_step_reward(observation: AuctionObservation, action: AuctionAction) -> float:
    if action.skip:
        return 0.02  # tiny budget-preservation nudge
    floor = max(observation.floor_price, 1e-3)
    if action.bid_amount < floor:
        return -0.05
    if action.bid_amount > observation.daily_budget_remaining + 0.1:
        return -0.5  # invalid bid (over budget)
    market = (sum(observation.recent_clearing_prices) / len(observation.recent_clearing_prices)
              if observation.recent_clearing_prices else floor + 0.2)
    win_prob = max(0.0, min(1.0, (action.bid_amount - market) / max(market, 1e-3) + 0.5))
    expected_clearing = max(floor, market)
    expected_revenue = 1.5 * win_prob * 0.18  # base CTR ~ 0.18 in the synthetic engine
    expected_clearing_paid = win_prob * expected_clearing
    margin = expected_revenue - expected_clearing_paid
    pacing_bonus = 0.05 if observation.spent_so_far_today < observation.spent_so_far_today + observation.daily_budget_remaining * 0.5 else 0.0
    return float(margin + pacing_bonus)

def advertiser_reward_fn(prompts, completions, observation_json, **_):
    rewards: List[float] = []
    for completion, obs_json in zip(completions, observation_json):
        text = completion if isinstance(completion, str) else completion[-1].get('content', '')
        observation = AuctionObservation.model_validate_json(obs_json)
        action = parse_llm_advertiser_action(text, n_creatives=len(observation.available_creatives))
        rewards.append(_per_step_reward(observation, action))
    return rewards

# Sanity-check on the validation set.
_demo_obs = AuctionObservation.model_validate_json(val_ds[0]['observation_json'])
_demo_action = parse_llm_advertiser_action('{"skip": false, "bid_amount": 0.8, "creative_id": 0}', n_creatives=len(_demo_obs.available_creatives))
print('demo per-step reward:', _per_step_reward(_demo_obs, _demo_action))

## 8. Smoke-test logging on first 2 GRPO steps

**Critical insurance step.** Verify W&B + per-step custom metrics fire correctly before launching the long run. A broken logger costs 4 hours of GPU time.

In [ ]:
wandb.login()  # paste API key when prompted

def _validation_metrics(training_step: int) -> dict:
    """Compute weekly_roas + bid_precision + budget_depletion_day + fatigue_sensitivity
    on a small validation episode every 10 steps."""
    if training_step % 10 != 0:
        return {}
    cfg = ARENA_TASKS[scheduler.current_tier]
    rng = random.Random(10_000 + training_step)
    opponents = _sample_persona_slate(rng, list(PERSONAS.keys())[:cfg.n_personas], n=cfg.n_personas)
    # Use the same mock policy for now — when the model is exported via FastLanguageModel.for_inference,
    # swap this for a wrapped LLM bid function.
    result = run_episode(seed=10_000 + training_step, cfg=cfg,
                         trained_policy=_MockTrainedPolicy().bid, opponents=opponents)
    return {
        'weekly_roas': result.weekly_roas,
        'bid_precision': result.bid_precision,
        'budget_depletion_day': result.budget_depletion_day,
        'fatigue_sensitivity': result.fatigue_sensitivity,
        'episode_return_total': result.weekly_revenue - result.weekly_spend,  # used by curriculum scheduler
    }

callback = make_arena_callback(
    run_name=RUN_NAME,
    wandb_project=WANDB_PROJECT,
    log_dir='logs',
    checkpoint_root='checkpoints',
    episode_dump_every=10,
    save_checkpoint_every=SAVE_EVERY_STEPS,
    best_metric_name='weekly_roas',
    higher_is_better=True,
    custom_metrics_fn=_validation_metrics,
    use_wandb=True,
    wandb_config={
        'base_model': BASE_MODEL,
        'lora_rank': LORA_RANK,
        'num_generations': NUM_GENERATIONS,
        'learning_rate': LEARNING_RATE,
        'max_new_tokens': MAX_NEW_TOKENS,
        'train_examples': len(train_ds),
        'val_examples': len(val_ds),
        'max_steps': MAX_STEPS,
        'curriculum_tier_initial': scheduler.current_tier,
    },
)
print('Callback ready.')

## 9. GRPO trainer + run

Two callbacks: the unified observability callback (W&B + CSV + JSONL + best-tracker) and the curriculum scheduler bridge that promotes the env tier on the fly.

In [ ]:
from trl import GRPOConfig, GRPOTrainer

config = GRPOConfig(
    output_dir=str(OUTPUT_DIR),
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    num_generations=NUM_GENERATIONS,
    max_prompt_length=MAX_SEQ_LEN - MAX_NEW_TOKENS,
    max_completion_length=MAX_NEW_TOKENS,
    max_steps=MAX_STEPS,
    save_steps=SAVE_EVERY_STEPS,
    logging_steps=LOG_EVERY_STEPS,
    bf16=False,
    fp16=True,
    optim='adamw_8bit',
    report_to=['wandb'],
    remove_unused_columns=False,
    seed=SEED,
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[advertiser_reward_fn],
    args=config,
    train_dataset=train_ds,
)
trainer.add_callback(callback)
trainer.add_callback(scheduler.as_callback())
trainer.train()

## 10. Validation pass + best-checkpoint selection

Best checkpoint = max mean `weekly_roas` over the last 5 validation episodes. The callback already snapshots `checkpoints/advertiser_run/best/` whenever a higher value is seen during training; this cell is the explicit final dump that the eval + HF Hub push read from.

In [ ]:
FastLanguageModel.for_inference(model)

def _generate(prompt: str) -> str:
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    out = model.generate(
        **inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
        pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
    )
    return tokenizer.decode(out[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True)

weekly_roas_samples: List[float] = []
for example in val_ds.select(range(min(5, len(val_ds)))):
    text = _generate(example['prompt'])
    obs = AuctionObservation.model_validate_json(example['observation_json'])
    action = parse_llm_advertiser_action(text, n_creatives=len(obs.available_creatives))
    weekly_roas_samples.append(_per_step_reward(obs, action))
mean_recent = statistics.mean(weekly_roas_samples) if weekly_roas_samples else 0.0
print(f'Final-pass mean per-step reward over {len(weekly_roas_samples)} samples: {mean_recent:.3f}')
wandb.log({'final/mean_per_step_reward': mean_recent})

FINAL_PATH = OUTPUT_DIR / 'final'
model.save_pretrained(str(FINAL_PATH))
tokenizer.save_pretrained(str(FINAL_PATH))
print(f'Saved final checkpoint to {FINAL_PATH}')
print(f'Best (auto-tracked) at:   {OUTPUT_DIR / "best"}')

## 11. Run the 3-mode advertiser eval (standard / edge / selfplay)

Mirrors `python -m scripts.advertiser_eval`. Uses the **best** checkpoint as the trained policy and **final** as the v1 selfplay opponent (or pass an earlier `checkpoint-10/` snapshot for stronger v2-vs-v1 contrast).

In [ ]:
from scripts.advertiser_eval import run_advertiser_eval

EVAL_RESULTS_PATH = Path('results/advertiser_eval.json')
EVAL_RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)

payload = run_advertiser_eval(
    task_name='arena_hard',
    checkpoint=None,  # set to str(OUTPUT_DIR / 'best') after training to use the trained model
    opponent_checkpoint=None,
    n_standard=20,
    n_edge_per_sub=10,
    n_selfplay=10,
    out_path=EVAL_RESULTS_PATH,
    include_baselines=True,
)
for mode, m in payload['per_mode'].items():
    print(f"[eval] {mode:>9s}  weekly_roas={m['weekly_roas_mean']:.3f}  bid_precision={m['bid_precision_mean']:.3f}  depl_day={m['budget_depletion_day_mean']:.2f}  fatigue_sens={m['fatigue_sensitivity_mean']:.3f}")

## 12. Generate the 5 advertiser PNGs

Pure CSV → PNG; matches the `make_plots.py` invocation in the eval script. PNGs land in `assets/plots/` and the source CSVs in `assets/data/`.

In [ ]:
from scripts.make_plots import (
    plot_bid_precision_hist,
    plot_budget_depletion_comparison,
    plot_fatigue_sensitivity,
    plot_loss_curve,
    plot_reward_curve,
)

PLOTS_DIR = Path('assets/plots'); PLOTS_DIR.mkdir(parents=True, exist_ok=True)
advertiser_csv = next(Path('logs').glob(f'training_run_{RUN_NAME}_*.csv'), None)
if advertiser_csv is None:
    print('No training CSV found; using empty placeholders.')
    advertiser_csv = Path('logs/_missing.csv')

for fn, kwargs in [
    (plot_reward_curve,        dict(advertiser_csv=advertiser_csv, baseline_csvs={}, out_path=PLOTS_DIR / 'reward_curve.png')),
    (plot_loss_curve,          dict(advertiser_csv=advertiser_csv, out_path=PLOTS_DIR / 'loss_curve.png')),
    (plot_bid_precision_hist,  dict(eval_json=EVAL_RESULTS_PATH, advertiser_csv=advertiser_csv, out_path=PLOTS_DIR / 'bid_precision_hist.png')),
    (plot_budget_depletion_comparison, dict(eval_json=EVAL_RESULTS_PATH, out_path=PLOTS_DIR / 'budget_depletion_comparison.png')),
    (plot_fatigue_sensitivity, dict(advertiser_csv=advertiser_csv, out_path=PLOTS_DIR / 'fatigue_sensitivity_corr.png')),
]:
    out = fn(**kwargs)
    print(f'  -> {out}')

## 13. Push to HuggingFace Hub

Auto-populated model card includes: training config, env config, reward decomposition formula, eval table from `advertiser_eval.json`, public W&B URL, training cost (T4-hours).

In [ ]:
from huggingface_hub import HfApi, login
login()  # paste token

BEST_PATH = OUTPUT_DIR / 'best'
model.push_to_hub(HF_HUB_REPO_ID, private=False)
tokenizer.push_to_hub(HF_HUB_REPO_ID, private=False)

_eval_summary = '\n'.join(
    f"- {mode}: weekly_roas={m['weekly_roas_mean']:.3f}, bid_precision={m['bid_precision_mean']:.3f}, depl_day={m['budget_depletion_day_mean']:.2f}, fatigue_sens={m['fatigue_sensitivity_mean']:.3f}"
    for mode, m in payload.get('per_mode', {}).items()
)

model_card = textwrap.dedent(f"""\
---
language: en
license: apache-2.0
library_name: peft
tags:
- grpo
- unsloth
- multi-agent
- ad-tech
- long-horizon
base_model: {BASE_MODEL}
---

# AdMarket Arena — Trained Advertiser

GRPO-trained LoRA adapter for the AdMarket Arena env's *advertiser* role. Bids, paces budget, and rotates creatives across a 7-day campaign in a 5-advertiser second-price auction.

**Training**: {MAX_STEPS} GRPO steps, num_generations={NUM_GENERATIONS}, lr={LEARNING_RATE}, on Colab T4. Reward = composable rubrics (per-step engagement + daily ROAS + weekly ROAS); see master plan Section 4.

**Eval (Section 7.2 robustness table)**:
{_eval_summary}

**W&B**: https://wandb.ai/<your-org>/{WANDB_PROJECT}
**Companion oversight model**: <your-org>/admarket-oversight-qwen2.5-3b-grpo (Plan 2)
**Eval JSON**: see `results/advertiser_eval.json` in the env repo, generated by `scripts/advertiser_eval.py`.
""")
Path('MODEL_CARD.md').write_text(model_card)
HfApi().upload_file(path_or_fileobj='MODEL_CARD.md', path_in_repo='README.md', repo_id=HF_HUB_REPO_ID, repo_type='model')
print('Pushed to', HF_HUB_REPO_ID)
wandb.finish()